# Create and Fill Bank Monitoring Database

This notebook renders the bank monitoring DDL with Russian comments, creates the SQLite database through the project pipeline, and previews the first rows from each generated table.

In [1]:
from contextlib import redirect_stdout
import io
from pathlib import Path
import sqlite3
import sys
from IPython.display import Markdown, display
import pandas as pd

from src.config import PROJECT_ROOT
from src.dataset import (  # noqa: E402
    BANK_SCHEMA_MAPPING_PATH,
    build_bank_monitoring_database,
    render_bank_monitoring_ddl,
)


COMMENT_STYLE = "inline"
COMMENT_VARIANT = "short"
PREVIEW_ROWS = 5

In [2]:
ddl = render_bank_monitoring_ddl(
    comment_style=COMMENT_STYLE,
    comment_variant=COMMENT_VARIANT,
)

print(ddl)

CREATE TABLE btm_cst ( -- Клиенты банка.
    c01 INTEGER, -- Идентификатор клиента.
    c02 TEXT   , -- Имя клиента.
    c03 TEXT   , -- Адрес клиента.
    c04 TEXT   , -- Код штата клиента.
    c05 TEXT     -- Телефон клиента.
);

CREATE TABLE btm_cex ( -- Экспортная копия клиентов.
    x01 TEXT   , -- Идентификатор клиента в выгрузке.
    x02 TEXT   , -- Имя клиента в выгрузке.
    x03 TEXT   , -- Адрес клиента в выгрузке.
    x04 TEXT   , -- Код штата в выгрузке.
    x05 TEXT     -- Телефон в выгрузке.
);

CREATE TABLE btm_accd ( -- Детали банковских счетов.
    a01 INTEGER, -- Идентификатор владельца счета.
    a02 TEXT   , -- Номер счета.
    a03 TEXT   , -- Тип счета.
    a04 INTEGER, -- Баланс счета.
    a05 TEXT   , -- Статус счета.
    a06 TEXT     -- Тип связи клиента со счетом.
);

CREATE TABLE btm_accs ( -- Срез банковских счетов.
    s01 INTEGER, -- Идентификатор клиента.
    s02 TEXT   , -- Номер счета.
    s03 TEXT   , -- Тип счета.
    s04 INTEGER, -- Текущий баланс.
  

In [3]:
db_path = build_bank_monitoring_database(
    comment_style=COMMENT_STYLE,
    comment_variant=COMMENT_VARIANT,
    overwrite=True,
)

print(f"SQLite database created: {db_path}")

schema_mapping = pd.read_json(BANK_SCHEMA_MAPPING_PATH, orient="index")
table_names = schema_mapping.index.tolist()

with sqlite3.connect(db_path) as connection:
    for table_name in table_names:
        row_count = connection.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        display(Markdown(f"## `{table_name}` ({row_count} rows)"))
        display(pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT {PREVIEW_ROWS}", connection))

SQLite database created: /home/dima/Sber/cursor_projects/vanna-sql/data/processed/bank_transaction_monitoring/bank_transaction_monitoring_inline_short.sqlite.db


## `btm_cst` (50 rows)

,c01,c02,c03,c04,c05
0,123001,Oliver,"200-1, Emeryville",CA,1897614000
1,123002,George,"201-2, New Brighton",MN,1897614137
2,123003,Harry,"202-3, Walnut Creek",NY,1897614274
3,123004,Jack,"203-4, Concord",TX,1897614411
4,123005,Jacob,"204-5, Mission Dist",WA,1897614548


## `btm_cex` (50 rows)

,x01,x02,x03,x04,x05
0,123001,Oliver,"200-1, Emeryville",CA,1897614000
1,123002,George,"201-2, New Brighton",MN,1897614137
2,123003,Harry,"202-3, Walnut Creek",NY,1897614274
3,123004,Jack,"203-4, Concord",TX,1897614411
4,123005,Jacob,"204-5, Mission Dist",WA,1897614548


...

,t01,t02,t03,t04,t05
0,4000-1900-3000,-1000.0,ATM withdrawal,NY,2020-01-05
1,5000-1700-5000,-2500.0,POS-Walmart,TX,2020-01-10
2,9000-1700-7000-4300,-4000.0,UPI transfer,WA,2020-01-15
3,4000-1901-3001,-5500.0,Bankers cheque,IL,2020-01-20
4,5000-1701-5001,7000.0,Net banking,AZ,2020-01-25


## `btm_msg` (50 rows)

,m01,m02,m03
0,Adhoc,All bank branches are closed due to a national...,mobile
1,Transaction Limit,Only limited withdrawals per card are allowed ...,mobile
2,Card Security,Confirm recent card activity if the transactio...,sms
3,Balance Alert,Your account balance crossed the configured mo...,email
4,Rate Update,Interest rate has been updated for selected de...,email


## `btm_rate` (100 rows)

,i01,i02,i03,i04
0,SAVINGS,0.036,01,2020
1,RECURRING DEPOSITS,0.051,01,2020
2,PRIVILEGED_INTEREST_RATE,0.066,01,2020
3,Credit Card,0.081,01,2020
4,SAVINGS,0.037,02,2020


In [4]:
from src.vanna_connector import initialize_vanna

/home/dima/Sber/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from qdrant_client import QdrantClient
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL, OPENAI_API_URL,\
     DENSE_EMBEDDING_MODEL_PATH

client = QdrantClient(url="http://127.0.0.1:6333",
                      api_key="12345")

/tmp/ipykernel_22368/103737403.py:5: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url="http://127.0.0.1:6333",


In [18]:
db_path

PosixPath('/home/dima/Sber/cursor_projects/vanna-sql/data/processed/bank_transaction_monitoring/bank_transaction_monitoring_inline_short.sqlite.db')

In [19]:
sqlite_config = {
    "params": {
        "url": str(db_path) # /home/dima/Sber/cursor_projects/vanna-sql/data/processed/bank_transaction_monitoring/bank_transaction_monitoring_inline_short.sqlite.db
    },
    "type": "sqlite"}
qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": "cpu"}
openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [16]:
DENSE_EMBEDDING_MODEL_PATH

'/home/dima/Sber/cursor_projects/vanna-sql/models/e5_large'

In [20]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5724.34it/s]
/home/dima/Sber/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


In [21]:
vanna_client.generate_sql("Как зовут всех клиентов банка?")

SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. ===Response Guidelines \n1. If the provided context is sufficient, please generate a valid SQL query without any explanations for the question. \n2. If the provided context is almost sufficient but requires knowledge of a specific string in a particular column, please generate an intermediate SQL query to find the distinct strings in that column. Prepend the query with a comment saying intermediate_sql \n3. If the provided context is insufficient, please explain why it can't be generated. \n4. Please use the most relevant table(s). \n5. If the question has been asked and answered before, please repeat the answer exactly as it was given before. \n6. Ensure that the output SQL is SQLite-compliant and executable, and free of syntax errors. \n"}, {'

'Недостаточно контекста: не предоставлена схема базы данных с таблицами и столбцами, где хранятся имена клиентов банка.'